In [1]:
import numpy as np
import pandas as pd
import random
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.datasets import imdb

positive_sentences = [
    "I love this product", "This movie was amazing", "The service was excellent",
    "I am very happy", "Fantastic experience", "Great quality", "Absolutely wonderful",
    "I enjoyed it", "Highly recommended", "Best purchase", "Very satisfying",
    "This made my day", "Superb performance", "I really like it", "Outstanding work",
    "It works perfectly", "Great support", "Very impressive", "Loved it", "Worth the money"
]

negative_sentences = [
    "I hate this product", "This movie was terrible", "The service was awful",
    "I am very disappointed", "Worst experience", "Poor quality", "Absolutely horrible",
    "I regret buying this", "Not recommended", "Waste of money", "Very unsatisfying",
    "This ruined my day", "Bad performance", "I dislike it", "Terrible work",
    "It does not work", "Very bad support", "Not impressive", "I hated it", "Totally useless"
]

data = []
for _ in range(300):
    data.append([random.choice(positive_sentences), 1])
    data.append([random.choice(negative_sentences), 0])

df = pd.DataFrame(data, columns=["text", "sentiment"])

tokenizer = Tokenizer(num_words=1000)
tokenizer.fit_on_texts(df["text"])
sequences = tokenizer.texts_to_sequences(df["text"])

X_dummy = pad_sequences(sequences, maxlen=10)
y_dummy = df["sentiment"].values

X_train_dummy, X_test_dummy, y_train_dummy, y_test_dummy = train_test_split(
    X_dummy, y_dummy, test_size=0.25, random_state=42
)

C:\Users\Varun\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


# 2. RNN on Dummy Data

In [2]:
rnn_dummy = Sequential([
    Embedding(input_dim=1000, output_dim=32),
    SimpleRNN(64, activation="tanh"),
    Dense(1, activation="sigmoid")
])

rnn_dummy.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
rnn_dummy.fit(X_train_dummy, y_train_dummy, epochs=2, validation_data=(X_test_dummy, y_test_dummy))

loss, acc_rnn_dummy = rnn_dummy.evaluate(X_test_dummy, y_test_dummy)
print("Simple RNN Accuracy on Dummy Data:", acc_rnn_dummy)

Epoch 1/2
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6200 - loss: 0.6755 - val_accuracy: 0.6800 - val_loss: 0.6386
Epoch 2/2
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8711 - loss: 0.5782 - val_accuracy: 0.9467 - val_loss: 0.4910
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9467 - loss: 0.4910 
Simple RNN Accuracy on Dummy Data: 0.9466666579246521


# 3. LSTM on Dummy Data

In [3]:
lstm_dummy = Sequential([
    Embedding(input_dim=1000, output_dim=32),
    LSTM(64),
    Dense(1, activation="sigmoid")
])

lstm_dummy.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
lstm_dummy.fit(X_train_dummy, y_train_dummy, epochs=2, validation_data=(X_test_dummy, y_test_dummy))

loss, acc_lstm_dummy = lstm_dummy.evaluate(X_test_dummy, y_test_dummy)
print("LSTM Accuracy on Dummy Data:", acc_lstm_dummy)

Epoch 1/2
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.4667 - loss: 0.6927 - val_accuracy: 0.6200 - val_loss: 0.6868
Epoch 2/2
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8533 - loss: 0.6804 - val_accuracy: 0.8800 - val_loss: 0.6671
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8800 - loss: 0.6671 
LSTM Accuracy on Dummy Data: 0.8799999952316284


# 4. Dataset Loading (IMDB)

In [4]:
max_features = 10000

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

x_train = pad_sequences(x_train)
x_test = pad_sequences(x_test)

# 5. RNN on IMDB Dataset

In [5]:
rnn_model = Sequential([
    Embedding(input_dim=max_features, output_dim=32),
    SimpleRNN(64, activation='tanh'),
    Dense(1, activation='sigmoid')
])

rnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

rnn_model.fit(x_train[:2000], y_train[:2000], epochs=3, batch_size=64, validation_split=0.2)

loss, acc_rnn_imdb = rnn_model.evaluate(x_test[:500], y_test[:500])
print("Simple RNN Accuracy on IMDB Dataset:", acc_rnn_imdb)

Epoch 1/3
25/25 ━━━━━━━━━━━━━━━━━━━━ 9s 341ms/step - accuracy: 0.5006 - loss: 0.6957 - val_accuracy: 0.5200 - val_loss: 0.6943
Epoch 2/3
25/25 ━━━━━━━━━━━━━━━━━━━━ 8s 326ms/step - accuracy: 0.7750 - loss: 0.6136 - val_accuracy: 0.5525 - val_loss: 0.6823
Epoch 3/3
25/25 ━━━━━━━━━━━━━━━━━━━━ 8s 322ms/step - accuracy: 0.9269 - loss: 0.4523 - val_accuracy: 0.5675 - val_loss: 0.6874
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.5200 - loss: 0.7156
Simple RNN Accuracy on IMDB Dataset: 0.5199999809265137


# 6. LSTM on IMDB Dataset

In [6]:
lstm_model = Sequential([
    Embedding(input_dim=max_features, output_dim=32),
    LSTM(64),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

lstm_model.fit(x_train[:2000], y_train[:2000], epochs=3, batch_size=64, validation_split=0.2)

loss, acc_lstm_imdb = lstm_model.evaluate(x_test[:500], y_test[:500])
print("LSTM Accuracy on IMDB Dataset:", acc_lstm_imdb)

Epoch 1/3
25/25 ━━━━━━━━━━━━━━━━━━━━ 15s 575ms/step - accuracy: 0.5194 - loss: 0.6917 - val_accuracy: 0.6025 - val_loss: 0.6877
Epoch 2/3
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 540ms/step - accuracy: 0.6144 - loss: 0.7025 - val_accuracy: 0.5225 - val_loss: 0.6812
Epoch 3/3
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 551ms/step - accuracy: 0.6794 - loss: 0.6324 - val_accuracy: 0.6750 - val_loss: 0.6505
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 110ms/step - accuracy: 0.6620 - loss: 0.6457
LSTM Accuracy on IMDB Dataset: 0.6620000004768372


# 7. Comparison of RNN and LSTM

In [ ]:
print("=== Final Comparison")
print("Dummy Data -> RNN:", acc_rnn_dummy, "| LSTM:", acc_lstm_dummy)
print("IMDB Data  -> RNN:", acc_rnn_imdb, "| LSTM:", acc_lstm_imdb)

=== Final Comparison ===
Dummy Data -> RNN: 0.9466666579246521 | LSTM: 0.8799999952316284
IMDB Data  -> RNN: 0.5199999809265137 | LSTM: 0.6620000004768372
